# Rewrite Agent Test Notebook

## Setup

In [ ]:
# Install required packages
# %pip install langchain langchain-openai openai python-dotenv

In [ ]:
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

model = os.getenv("LLM_MODEL_NAME", "gpt-4o-mini")

print(f"✅ OpenAI client initialized with model '{model}'")

In [ ]:
from utils.logging_utils import init_logs, log_openai_response, get_logs_dataframe, get_summary, list_log_files

RUN_ID = os.getenv("RUN_ID", None)

init_logs(RUN_ID)

print("✅ Logging initialized")
print(f"RUN_ID: {RUN_ID}")

In [ ]:
# Load the suggested changes JSON
import json

with open("../artifact/suggested_changes_bbc.json", "r") as f:
    suggested_changes = json.load(f)

print("✅ Loaded suggested_changes_bbc.json")
print(f"Status: {suggested_changes['status']}")
print(f"user_modified_content: {suggested_changes['user_modified_content'][:100] if suggested_changes['user_modified_content'] else '(empty)'}")

## Rewrite Function

In [ ]:
REWRITE_PROMPT = """You are a Senior Copy Editor. Your task is to rewrite the draft based on user modifications and reviewer feedback.

BASE DRAFT:
{ai_draft}

USER MODIFICATIONS (MANDATORY):
{user_modified_content}

REVIEWER FEEDBACK:
{reviewer_notes}

STYLE DNA:
{style_dna}

INSTRUCTIONS:
1. Apply USER MODIFICATIONS exactly as requested
2. Address the REVIEWER FEEDBACK violations
3. Maintain the original facts (e.g., numbers, names, dates)
4. Follow the STYLE DNA constraints
5. Output ONLY the final rewritten content
"""

In [ ]:
from langchain_openai import ChatOpenAI

# Create LangChain LLM
llm = ChatOpenAI(model=model, temperature=0.3)

def rewrite_content(suggested_changes: dict) -> dict:
    """
    Rewrite content based on user modifications and reviewer feedback.
    
    Args:
        suggested_changes: The JSON with ai_draft, user_modified_content, reviewer_notes, style_dna
    
    Returns:
        Dictionary with rewritten content and safety checks
    """
    # Extract data
    ai_draft = suggested_changes.get("ai_draft", "")
    user_modifications = suggested_changes.get("user_modified_content", "")
    reviewer_notes = suggested_changes.get("reviewer_notes", {})
    style_dna = suggested_changes.get("style_dna", "")
    
    # Build prompt
    prompt = REWRITE_PROMPT.format(
        ai_draft=ai_draft,
        user_modified_content=user_modifications if user_modifications else "No user modifications provided",
        reviewer_notes=json.dumps(reviewer_notes, indent=2),
        style_dna=style_dna
    )
    
    # Call LLM
    response = client.responses.create(
        model=model,
        input=prompt
    )
    
    rewritten_content = response.output_text
    
    # Safety checks
    
    # Fact check: ensure key facts are preserved
    fact_check_passed = True
    key_facts = ["128", "quantum", "NovaQ"]  # Add more as needed
    for fact in key_facts:
        if fact.lower() not in rewritten_content.lower():
            fact_check_passed = False
            break
    
    # Markdown check
    markdown_check_passed = True
    if "[" in rewritten_content and "]" in rewritten_content:
        markdown_check_passed = False
    
    # Log the call
    task_name = "BBC_Rewriter"
    log_openai_response(response, prompt, task_name=task_name, run_id=RUN_ID)
    
    return {
        "rewritten_content": rewritten_content,
        "fact_check_passed": fact_check_passed,
        "markdown_check_passed": markdown_check_passed
    }

print("✅ Rewrite function defined")

## Run Rewrite

In [ ]:
# Run rewrite
rewrite_result = rewrite_content(suggested_changes)

print("=" * 50)
print("REWRITE RESULT")
print("=" * 50)
print(f"Fact Check Passed: {rewrite_result['fact_check_passed']}")
print(f"Markdown Check Passed: {rewrite_result['markdown_check_passed']}")
print("\n--- REWRITTEN CONTENT ---\n")
print(rewrite_result['rewritten_content'])

## Save Result

In [ ]:
# Save the rewritten content

output_data = {
    "run_id": suggested_changes["run_id"],
    "original_draft": suggested_changes["ai_draft"],
    "user_modifications": suggested_changes["user_modified_content"],
    "rewritten_content": rewrite_result["rewritten_content"],
    "fact_check_passed": rewrite_result["fact_check_passed"],
    "markdown_check_passed": rewrite_result["markdown_check_passed"]
}

# Save to artifact folder
os.makedirs("../artifact", exist_ok=True)

with open("../artifact/rewritten_bbc.json", "w") as f:
    json.dump(output_data, f, indent=2)

print("✅ Saved to artifact/rewritten_bbc.json")